# METRIC-GEE NaN Debug — Qadam ba Qadam
Idaho WRS Path=40 Row=30 | 2024-07-23

Har bir qadam natijasini Map da ko'ramiz va valid piksel sonini hisoblaymiz.

In [20]:
import ee
import geemap
import sys, os
sys.path.insert(0, r'D:/Cloud_comp/METRC/metric_gee_v2.1')
ee.Initialize()
print('GEE OK')

GEE OK


In [21]:
# ── Yordamchi funksiyalar ──────────────────────────────────
def stats(img, label, geometry, scale=300):
    """Min / Mean / Max va valid piksel soni."""
    bands = img.bandNames().getInfo()
    band  = bands[0] if bands else 'constant'

    res = img.reduceRegion(
        ee.Reducer.min().combine(ee.Reducer.mean(), sharedInputs=True)
                        .combine(ee.Reducer.max(),  sharedInputs=True)
                        .combine(ee.Reducer.count(), sharedInputs=True),
        geometry, scale, maxPixels=1e9
    ).getInfo()

    mn  = res.get(f'{band}_min',   res.get('min',   '?'))
    mu  = res.get(f'{band}_mean',  res.get('mean',  '?'))
    mx  = res.get(f'{band}_max',   res.get('max',   '?'))
    cnt = res.get(f'{band}_count', res.get('count', '?'))

    print(f'[{label:20s}]  min={mn:.3f}  mean={mu:.3f}  max={mx:.3f}  n={cnt}')
    return cnt

# NaN tracker — har qadam uchun n ni saqlaymiz
nan_track = {}

def track(img, label, geometry, scale=300):
    n = stats(img, label, geometry, scale)
    nan_track[label] = n
    return n

## 0. Tile va Xarita

In [22]:
WRS_PATH, WRS_ROW = 40, 30
DATE = '2024-07-23'

# Tile geometry
dummy    = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
            .filter(ee.Filter.eq('WRS_PATH', WRS_PATH))
            .filter(ee.Filter.eq('WRS_ROW',  WRS_ROW))
            .first())
GEOMETRY = dummy.geometry()

# Landsat image
RAW = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
       .filterBounds(GEOMETRY)
       .filterDate(DATE, '2024-07-24')
       .filter(ee.Filter.eq('WRS_PATH', WRS_PATH))
       .filter(ee.Filter.eq('WRS_ROW',  WRS_ROW))
       .first())

print('Image:', RAW.id().getInfo())

# Xarita
Map = geemap.Map()
Map.centerObject(GEOMETRY, 9)
Map

Image: LC08_040030_20240723


Map(center=[43.1848751638689, -113.59333062313155], controls=(WidgetControl(options=['position', 'transparent_…

## 1. Raw SR bands — masksiz piksel soni

In [23]:
# Raw B4 (Red) piksel soni — hech qanday mask yo'q
raw_b4 = RAW.select('SR_B4')
track(raw_b4, 'Raw SR_B4', GEOMETRY, 100)

Map.addLayer(raw_b4.clip(GEOMETRY),
             {'min':5000, 'max':25000, 'palette':['black','white']},
             '1. Raw SR_B4')
print('Raw band qoshildi')

[Raw SR_B4           ]  min=879.000  mean=11951.849  max=53666.000  n=2531823
Raw band qoshildi


## 2. Scale factors + Rename bands

In [24]:
from metric_gee.inputs.landsat import LandsatLoader
from metric_gee.config.settings import Settings

cfg = Settings(wrs_path=WRS_PATH, wrs_row=WRS_ROW)
landsat = LandsatLoader(cfg)

img_scaled = landsat._apply_scale_factors(RAW)
img_scaled = ee.Image(img_scaled.copyProperties(RAW, RAW.propertyNames()))
img_renamed = landsat._rename_bands(img_scaled)

b_red = img_renamed.select('B_RED')
track(b_red, 'Scale+Rename B_RED', GEOMETRY, 100)

Map.addLayer(b_red.clip(GEOMETRY),
             {'min':0, 'max':0.3, 'palette':['black','white']},
             '2. B_RED scaled')
print('OK')

[Scale+Rename B_RED  ]  min=0.000  mean=0.129  max=1.000  n=2531823
OK


## 3. Cloud mask — qancha piksel ketadi?

In [25]:
img_cloud = landsat._apply_cloud_mask(img_renamed)
b_red_cloud = img_cloud.select('B_RED')

n_raw   = nan_track.get('Scale+Rename B_RED', 1)
n_cloud = track(b_red_cloud, 'After cloud mask', GEOMETRY, 100)

print(f'Cloud mask olib tashladi: {n_raw - n_cloud} piksel '
      f'({(n_raw-n_cloud)/n_raw*100:.1f}%)')

# Cloud mask o'zi nima qiladi?
qa = RAW.select('QA_PIXEL')
cloud_bit  = qa.bitwiseAnd(1 << 3).neq(0)
shadow_bit = qa.bitwiseAnd(1 << 4).neq(0)
cloud_mask = cloud_bit.Or(shadow_bit)

Map.addLayer(cloud_mask.clip(GEOMETRY).selfMask(),
             {'palette':['red']}, '3a. Cloud+Shadow mask')
Map.addLayer(b_red_cloud.clip(GEOMETRY),
             {'min':0, 'max':0.3, 'palette':['black','white']},
             '3b. B_RED after cloud mask')
print('OK')

[After cloud mask    ]  min=0.000  mean=0.121  max=0.655  n=2408409
Cloud mask olib tashladi: 123414 piksel (4.9%)
OK


## 4. NDVI, LAI — zom uchun asosiy kirish

In [26]:
from metric_gee.core.vegetation import VegetationIndices
veg = VegetationIndices(cfg)

ndvi, savi, lai = veg.calc_all(img_cloud)

track(ndvi, 'NDVI', GEOMETRY, 100)
track(lai,  'LAI',  GEOMETRY, 100)

# LAI = 0 piksellar soni (zom = ZOM_MIN = 0.005 bo'ladi)
lai_zero = lai.eq(0)
cnt_zero = lai_zero.reduceRegion(
    ee.Reducer.sum(), GEOMETRY, 100, maxPixels=1e9).getInfo()
print(f'LAI == 0 piksellar: {cnt_zero}')

Map.addLayer(ndvi.clip(GEOMETRY),
             {'min':-0.3, 'max':0.9, 'palette':['brown','white','green']},
             '4a. NDVI')
Map.addLayer(lai.clip(GEOMETRY),
             {'min':0, 'max':6, 'palette':['white','green']},
             '4b. LAI')
print('OK')

[NDVI                ]  min=-0.741  mean=0.340  max=1.000  n=2408409
[LAI                 ]  min=0.000  mean=0.842  max=6.000  n=2408409
LAI == 0 piksellar: {'LAI': 51708.282352941176}
OK


## 5. Albedo

In [27]:
from metric_gee.core.albedo import AlbedoCalculator
alb_calc = AlbedoCalculator(cfg)
albedo = alb_calc.calc_albedo(img_cloud)

track(albedo, 'Albedo', GEOMETRY, 100)

Map.addLayer(albedo.clip(GEOMETRY),
             {'min':0.05, 'max':0.35, 'palette':['black','white']},
             '5. Albedo')
print('OK')

[Albedo              ]  min=0.028  mean=0.165  max=0.468  n=2408406
OK


## 6. Ts, TsDEM

In [28]:
from metric_gee.core.surface_temp import SurfaceTemperature
from metric_gee.inputs.dem import DEMLoader

dem_loader = DEMLoader(cfg)
dem = dem_loader.get_terrain(GEOMETRY)
z_datum = dem_loader.get_mean_elevation(GEOMETRY)

st = SurfaceTemperature(cfg)
ts     = st.calc_ts(img_cloud)
ts_dem = st.calc_ts_dem(ts, dem, z_datum)

track(ts,     'Ts',     GEOMETRY, 100)
track(ts_dem, 'TsDEM',  GEOMETRY, 100)

Map.addLayer(ts.clip(GEOMETRY),
             {'min':285, 'max':340, 'palette':['blue','yellow','red']},
             '6. Ts (K)')
print('OK')

[Ts                  ]  min=267.492  mean=319.875  max=340.173  n=2408409
[TsDEM               ]  min=263.037  mean=320.102  max=339.036  n=2408409
OK


## 7. Rn — Net Radiation

In [29]:
from metric_gee.core.emissivity import EmissivityCalculator
from metric_gee.core.net_radiation import NetRadiation
from metric_gee.inputs.era5 import ERA5Loader
from metric_gee.utils.solar_geometry import SolarGeometry

emiss = EmissivityCalculator()
rn_calc = NetRadiation(cfg)
era5_loader = ERA5Loader(cfg)
solar_calc  = SolarGeometry()

img_date = ee.Date(RAW.get('system:time_start'))
sun_elev = ee.Number(RAW.get('SUN_ELEVATION'))

e0, enb = emiss.calc_emissivity(lai, ndvi)
era5_h  = era5_loader.get_hourly(GEOMETRY, img_date)
solar   = solar_calc.calc_solar_params(DATE, GEOMETRY, sun_elev)
rn, tau_sw = rn_calc.calc_rn(albedo, ts, e0, era5_h, dem, solar)

track(rn, 'Rn', GEOMETRY, 100)

Map.addLayer(rn.clip(GEOMETRY),
             {'min':200, 'max':800, 'palette':['blue','yellow','red']},
             '7. Rn (W/m2)')
print('OK')

[Rn                  ]  min=238.793  mean=457.745  max=802.009  n=3300707
OK


## 8. G — Soil Heat Flux

In [30]:
from metric_gee.core.soil_heat_flux import SoilHeatFlux
g_calc = SoilHeatFlux(cfg)
g = g_calc.calc_g(rn, ts, ndvi, albedo, lai)

track(g, 'G', GEOMETRY, 100)

Map.addLayer(g.clip(GEOMETRY),
             {'min':0, 'max':300, 'palette':['blue','yellow','red']},
             '8. G (W/m2)')
print('OK')

[G                   ]  min=0.000  mean=99.996  max=353.602  n=2408406
OK


## 9. Aerodynamics — zom, u*, rah

**Eng muhim qadam** — zom NaN bo'lsa u* va rah ham NaN bo'ladi.

In [13]:
from metric_gee.core.aerodynamics import Aerodynamics
aero_calc = Aerodynamics(cfg)

u200    = aero_calc.calc_u200(era5_h, dem)
zom     = aero_calc.calc_zom(lai, ndvi, albedo, dem)
u_star  = aero_calc.calc_ustar(u200, zom)
rah     = aero_calc.calc_rah_neutral(u_star)
rho_air = aero_calc.calc_rho_air(dem, ts)

track(u200,    'u200',    GEOMETRY, 100)
track(zom,     'zom',     GEOMETRY, 100)
track(u_star,  'u_star',  GEOMETRY, 100)
track(rah,     'rah',     GEOMETRY, 100)
track(rho_air, 'rho_air', GEOMETRY, 100)

# zom NaN bormi? — LAI > 0 bo'lsa zom > ZOM_MIN = 0.005
zom_nan = zom.unmask(-1).eq(-1)
cnt_zom_nan = zom_nan.reduceRegion(
    ee.Reducer.sum(), GEOMETRY, 100, maxPixels=1e9).getInfo()
print(f'zom NaN piksel soni: {cnt_zom_nan}')

Map.addLayer(zom.clip(GEOMETRY),
             {'min':0.005, 'max':0.15, 'palette':['white','green']},
             '9a. zom (m)')
Map.addLayer(rah.clip(GEOMETRY),
             {'min':10, 'max':100, 'palette':['blue','red']},
             '9b. rah (s/m)')
print('OK')

[u200                ]  min=2.000  mean=2.163  max=3.697  n=4587953
[zom                 ]  min=0.005  mean=0.018  max=0.243  n=2405719
[u_star              ]  min=0.077  mean=0.092  max=0.201  n=3297834
[rah                 ]  min=36.277  mean=81.693  max=94.422  n=3297834
[rho_air             ]  min=0.729  mean=0.906  max=1.057  n=3300954
zom NaN piksel soni: {'zom': 596544.9647058822}
OK


## 10. H — pipeline._run_cimec + calc_h_iterative

In [15]:
from pipeline import METRICPipeline

# Pipeline yaratish — barcha modullar ichida
pipe = METRICPipeline(settings=cfg)
pipe.geometry  = GEOMETRY
pipe.verbose   = False
pipe.dem       = dem
pipe.elevation = dem.select('elevation')
pipe.z_datum   = z_datum

# ETr — pipeline._calc_etr_hourly aynan shu logikani ishlatadi
rs_down_mjhr = rn_calc._calc_rs_down(tau_sw, solar).multiply(0.0036)
etr_hourly   = pipe._calc_etr_hourly(era5_h, rs_down_mjhr, img_date)

aero = {'u200': u200, 'zom': zom, 'u_star': u_star, 'rah': rah, 'rho_air': rho_air}

# _run_cimec — pipeline.py dagi aynan o'sha metod
anchor_data, calib = pipe._run_cimec(ndvi, ts_dem, rn, g, aero, etr_hourly, GEOMETRY)
print(f"a={calib['a_val']:.4f}  b={calib['b_val']:.2f}")

# calc_h_iterative — pipeline.py dagi aynan o'sha metod
H = pipe.h_calc.calc_h_iterative(
    anchor_data, ts_dem, ts, aero, rn, g, GEOMETRY, verbose=True)

track(H, 'H', GEOMETRY, 100)

Map.addLayer(H.clip(GEOMETRY),
             {'min':-50, 'max':400, 'palette':['blue','white','red']},
             '10. H (W/m2)')
print('OK')

  LULC piksel soni: 4583101
  Cold LC kodlar: [40]   Hot LC kodlar: [60, 50]
  L1-CIMEC: cold=4215  hot=2945
  ✅ Anchor tanlandi: L1-CIMEC
a=1.2180  b=-369.35
    Iter  1: dT_hot=9.1164 K  rah_hot=25.275 s/m  a=0.39570  b=-121.421
    Iter  2: dT_hot=9.1164 K  rah_hot=25.275 s/m  a=0.42001  b=-129.442
    Iter  3: dT_hot=9.1164 K  rah_hot=25.275 s/m  a=0.45217  b=-140.051
    ✅ Converged at iter 3
[H                   ]  min=-55.351  mean=169.777  max=389.884  n=3298056
OK


## 11. LE, ETinst, ETrF — NaN asosiy joyi

In [31]:
from metric_gee.products.instantaneous_et import InstantaneousET
from metric_gee.products.daily_et import DailyET

et_inst_calc = InstantaneousET()
et_daily     = DailyET()

le, et_inst = et_inst_calc.calc(rn, g, H, ts)
etrf        = et_daily.calc_etrf(et_inst, etr_hourly)

track(le,      'LE',      GEOMETRY, 100)
track(et_inst, 'ETinst',  GEOMETRY, 100)
track(etrf,    'ETrF',    GEOMETRY, 100)

# ETrF NaN bormi?
etrf_nan = etrf.unmask(-1).eq(-1)
cnt_etrf_nan = etrf_nan.reduceRegion(
    ee.Reducer.sum(), GEOMETRY, 100, maxPixels=1e9).getInfo()
print(f'ETrF NaN piksel soni: {cnt_etrf_nan}')

Map.addLayer(etrf.clip(GEOMETRY),
             {'min':0, 'max':1.05, 'palette':['red','yellow','blue']},
             '11. ETrF (0..1.05)')
Map.addLayer(le.clip(GEOMETRY),
             {'min':-50, 'max':600, 'palette':['red','white','blue']},
             '11b. LE (W/m2)')
print('OK')

[LE                  ]  min=0.000  mean=187.905  max=819.366  n=3297829
[ETinst              ]  min=0.000  mean=0.280  max=1.179  n=3297829
[ETrF                ]  min=0.000  mean=0.377  max=1.050  n=3297778
ETrF NaN piksel soni: {'ETrF': 953135.6470588236}
OK


## 12. NaN Tahlil — qaysi qadam da eng ko'p piksel yo'qoldi?

In [32]:
print('\n=== NaN Tahlil ===')
print(f'{"Qadam":<25} {"n":>10}  {"delta":>10}')
print('-' * 48)

prev = None
for name, n in nan_track.items():
    if n is None:
        n = 0
    delta = (n - prev) if prev is not None else 0
    flag = '  <-- YO\'QOLDI' if delta < -100 else ''
    print(f'{name:<25} {n:>10}  {delta:>+10}{flag}')
    prev = n


=== NaN Tahlil ===
Qadam                              n       delta
------------------------------------------------
Raw SR_B4                    2531823          +0
Scale+Rename B_RED           2531823          +0
After cloud mask             2408409     -123414  <-- YO'QOLDI
NDVI                         2408409          +0
LAI                          2408409          +0
Albedo                       2408406          -3
Ts                           2408409          +3
TsDEM                        2408409          +0
Rn                           3300707     +892298
G                            2408406     -892301  <-- YO'QOLDI
LE                           3297829     +889423
ETinst                       3297829          +0
ETrF                         3297778         -51


## 13. Maxsus: ETrF = 0 vs NaN farqi

In [18]:
# ETrF 0 va NaN ni farqlash
# unmask(999) -> 999 edi NaN, boshqa qiymatlar valid
etrf_filled = etrf.unmask(999)

is_nan   = etrf_filled.eq(999)
is_zero  = etrf_filled.lte(0.01).And(is_nan.Not())
is_valid = is_nan.Not().And(is_zero.Not())

res = ee.Image.cat([
    is_nan.rename('nan_pixels'),
    is_zero.rename('zero_pixels'),
    is_valid.rename('valid_pixels'),
]).reduceRegion(
    ee.Reducer.sum(), GEOMETRY, 100, maxPixels=1e9
).getInfo()

total = sum(v for v in res.values() if v)
print(f'NaN piksellar:   {res["nan_pixels"]:>8}  ({res["nan_pixels"]/total*100:.1f}%)')
print(f'0 piksellar:     {res["zero_pixels"]:>8}  ({res["zero_pixels"]/total*100:.1f}%)')
print(f'Valid piksellar: {res["valid_pixels"]:>8}  ({res["valid_pixels"]/total*100:.1f}%)')

# Xaritada ko'rsatish
Map.addLayer(is_nan.selfMask().clip(GEOMETRY),
             {'palette':['red']}, '13. ETrF = NaN')
Map.addLayer(is_valid.selfMask().clip(GEOMETRY),
             {'palette':['green']}, '13b. ETrF valid')
print('OK')

NaN piksellar:   953135.6470588236  (22.8%)
0 piksellar:     968155.7607843692  (23.2%)
Valid piksellar: 2260717.3450988755  (54.1%)
OK


## 14. ESA WorldCover — Cropland NaN tahlili

In [19]:
# ESA WorldCover cropland (kod=40)
lulc = ee.Image('ESA/WorldCover/v100/2020').select('Map').clip(GEOMETRY)
cropland_mask = lulc.eq(40)

# Ekin yerlar ichida ETrF NaN bormi?
etrf_on_crop = etrf.updateMask(cropland_mask)
crop_nan = etrf_on_crop.unmask(999).eq(999).And(cropland_mask)

res2 = ee.Dictionary({
    'crop_total':   cropland_mask.reduceRegion(ee.Reducer.sum(), GEOMETRY, 100, maxPixels=1e9).values().get(0),
    'crop_nan_et':  crop_nan.reduceRegion(ee.Reducer.sum(), GEOMETRY, 100, maxPixels=1e9).values().get(0),
    'crop_valid_et':etrf_on_crop.reduceRegion(ee.Reducer.count(), GEOMETRY, 100, maxPixels=1e9).values().get(0),
}).getInfo()

print(f'Cropland jami piksel:      {res2["crop_total"]}')
print(f'Cropland ETrF NaN:         {res2["crop_nan_et"]}  '
      f'({res2["crop_nan_et"]/max(res2["crop_total"],1)*100:.1f}%)')
print(f'Cropland ETrF valid:       {res2["crop_valid_et"]}  '
      f'({res2["crop_valid_et"]/max(res2["crop_total"],1)*100:.1f}%)')

Map.addLayer(cropland_mask.selfMask().clip(GEOMETRY),
             {'palette':['yellow']}, '14a. Cropland (ESA)')
Map.addLayer(crop_nan.selfMask().clip(GEOMETRY),
             {'palette':['red']}, '14b. Cropland ETrF NaN')
Map.addLayer(etrf_on_crop.clip(GEOMETRY),
             {'min':0, 'max':1.05, 'palette':['red','yellow','blue']},
             '14c. ETrF on Cropland')
print('\nXaritani koring!')

Cropland jami piksel:      773953.0039215687
Cropland ETrF NaN:         141176.42352941175  (18.2%)
Cropland ETrF valid:       547779  (70.8%)

Xaritani koring!
